# Enhanced Sentiment Analysis for E-commerce Using Deep Learning

---

**Hierarchical Multi-Granularity Sentiment (HMGS) Model** — multi-level sentiment analysis (document, sentence, and aspect) on e-commerce reviews using BERT, attention mechanisms, and CRF-based aspect extraction.

**Notebook Structure:**
1. Environment Setup (Cells 1-3)
2. Data Acquisition & Preprocessing (Cells 4-6)
3. Model Architecture (Cells 7-8)
4. Training Protocol (Cells 9-10)
5. Evaluation Framework (Cell 11)
6. Baselines & Ablation (Cells 12-13)
7. Visualisation (Cell 14)
8. Execution & Results (Cells 15-21)

**3 Models Compared:** TF-IDF + LR | BERT+BiLSTM+Attention | HMGS (Ours)

---
## 1. Environment Setup

### Architecture

```
Review → Sentence Segmentation → [S₁, S₂, ..., Sₙ]
                                      │
                          (per sentence) BERT Encoder
                          → [CLS] + token embeddings
                                      │
                  ┌───────────────────┼───────────────────┐
                  ▼                   ▼                   ▼
           Aspect Extraction    Sentence Sentiment   Aspect Sentiment
           (BERT + CRF)        (CLS → FFN)         (ASP ⊕ CLS → FFN)
                                      │
                      Sentence [CLS] pooling
                              │
                  Document Attention Aggregation
                              │
                  Document Sentiment Prediction
```

### Multi-Task Loss (Staged Training)

**Phase 1** (Amazon Reviews): $\mathcal{L}_1 = \lambda_1 \mathcal{L}_{doc} + \lambda_2 \mathcal{L}_{sent}$

**Phase 2** (SemEval-2014):  $\mathcal{L}_2 = \lambda_3 \mathcal{L}_{ATE} + \lambda_4 \mathcal{L}_{ASC}$

Phase 1 trains document + sentence heads on Amazon reviews. Phase 2 fine-tunes ATE + ASC heads on SemEval-2014 gold data using the BERT encoder from Phase 1 (transfer learning).

In [ ]:
# ============================================================================
# CELL 1: Install dependencies
# ============================================================================
# NOTE: datasets<4 is pinned because McAuley-Lab/Amazon-Reviews-2023 uses a
# loading script that is not supported in datasets>=4.0.0.
# If you use datasets>=4, the notebook falls back to streaming JSONL directly.

!pip install -q torch>=2.1.0 transformers>=4.36.0 "datasets>=2.16.0,<4" \
    scikit-learn>=1.3.0 pandas>=2.1.0 numpy>=1.24.0 \
    matplotlib>=3.8.0 seaborn>=0.13.0 nltk>=3.8.0 \
    pytorch-crf>=0.7.2 scipy>=1.11.0 requests>=2.31.0

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Dependencies installed.")

In [ ]:
# ============================================================================
# CELL 2: Imports
# ============================================================================

import os, json, random, warnings
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchcrf import CRF

from transformers import (
    BertTokenizerFast,
    BertModel,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, accuracy_score, cohen_kappa_score,
)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline as SkPipeline
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import sent_tokenize

warnings.filterwarnings('ignore')

# --- Device setup (CUDA > MPS > CPU) ---
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE.type == "mps":
    print(f"GPU: Apple Silicon (MPS backend)")


def clear_gpu_cache():
    """Free GPU memory on any backend."""
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()

# --- Mixed precision setup (handles both old and new PyTorch) ---
try:
    from torch.amp import autocast, GradScaler
    AMP_DEVICE = DEVICE.type
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    AMP_DEVICE = None  # old API doesn't need device_type


def set_seed(seed: int):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def amp_autocast():
    """Return the correct autocast context manager for the PyTorch version."""
    if AMP_DEVICE is not None:
        return autocast(device_type=AMP_DEVICE)
    return autocast()


print("Imports complete.")

In [ ]:
# ============================================================================
# CELL 3: Configuration
# ============================================================================
# All hyperparameters in one place.
# For quick Colab test: set train_size=5000, val_size=1000, test_size=1000, seeds=[42]

@dataclass
class Config:
    # Model
    bert_model_name: str = "bert-base-uncased"
    num_sentiment_classes: int = 3       # positive, neutral, negative
    num_aspect_bio_tags: int = 3         # B-ASP, I-ASP, O
    classifier_dropout: float = 0.3
    hidden_dim: int = 768

    # Training
    learning_rate_bert: float = 2e-5
    learning_rate_heads: float = 1e-3
    weight_decay: float = 0.01
    max_epochs: int = 5
    batch_size: int = 16
    gradient_accumulation_steps: int = 2
    warmup_ratio: float = 0.1
    max_seq_length: int = 128            # 128 is enough for single sentences
    max_doc_sentences: int = 10
    early_stopping_patience: int = 2

    # Loss weights
    lambda_doc: float = 1.0
    lambda_sent: float = 0.5
    lambda_ate: float = 0.3
    lambda_asc: float = 0.7

    # Data — reduce these for quick testing
    train_size: int = 50_000
    val_size: int = 5_000
    test_size: int = 5_000
    amazon_categories: List[str] = field(
        default_factory=lambda: ["Electronics"]
    )

    # Reproducibility
    seeds: List[int] = field(default_factory=lambda: [42, 123, 456, 789, 1024])

    # Paths
    output_dir: str = "./outputs"
    checkpoint_dir: str = "./checkpoints"

    def save(self, path: str):
        with open(path, 'w') as f:
            json.dump(self.__dict__, f, indent=2, default=str)


config = Config()
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.checkpoint_dir, exist_ok=True)
config.save(os.path.join(config.output_dir, "config.json"))

print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")
print(f"Training samples: {config.train_size:,}")
print(f"Seeds: {config.seeds}")

---
## 2. Data Acquisition & Preprocessing

**Source:** Amazon Customer Reviews (McAuley Lab, 2023) via HuggingFace.  
**Sentiment mapping:** 1-2 stars → Negative (0), 3 stars → Neutral (1), 4-5 stars → Positive (2).  
**Known limitation:** 3-star reviews are noisy neutral indicators. Mitigated via class-weighted loss.

In [ ]:
# ============================================================================
# CELL 4: Load Amazon Reviews
# ============================================================================
# Strategy 1: HuggingFace datasets (library versions < 4.0)
# Strategy 2: Stream JSONL directly from HuggingFace Hub
# Strategy 3: Download from UCSD mirror (compressed JSONL.gz)
# Strategy 4: Synthetic demo data fallback

def load_amazon_reviews(categories: List[str], max_total: int) -> pd.DataFrame:
    """Load Amazon Reviews 2023 with multiple fallback strategies."""
    per_cat = max_total // len(categories)
    frames = []

    for cat in categories:
        print(f"Loading {cat} ({per_cat:,} reviews)...")
        df_cat = None

        # --- Strategy 1: HuggingFace datasets library (requires datasets<4) ---
        if df_cat is None:
            try:
                from datasets import __version__ as ds_version
                major = int(ds_version.split(".")[0])
                if major < 4:
                    ds = load_dataset(
                        "McAuley-Lab/Amazon-Reviews-2023",
                        f"raw_review_{cat}",
                        split=f"full[:{per_cat}]",
                        trust_remote_code=True,
                    )
                    df_cat = ds.to_pandas()
                    df_cat = df_cat.rename(columns={"text": "review_text", "rating": "star_rating"})
                    df_cat["category"] = cat
                    df_cat = df_cat[["review_text", "star_rating", "category"]]
                    print(f"  Strategy 1 (HF datasets): Loaded {len(df_cat):,} reviews.")
                else:
                    print(f"  Strategy 1 skipped: datasets v{ds_version} dropped loading-script support.")
                    df_cat = None
            except Exception as e:
                print(f"  Strategy 1 failed: {e}")
                df_cat = None

        # --- Strategy 2: Stream JSONL from HuggingFace Hub ---
        if df_cat is None:
            try:
                import requests as req
                url = ("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023"
                       f"/resolve/main/raw/review_categories/{cat}.jsonl")
                print(f"  Strategy 2: Streaming JSONL from HuggingFace Hub...")
                records = []
                with req.get(url, stream=True, timeout=120) as resp:
                    resp.raise_for_status()
                    for line in resp.iter_lines():
                        if line:
                            record = json.loads(line)
                            text = str(record.get("text", "")).strip()
                            if len(text) > 10:
                                records.append({
                                    "review_text": text,
                                    "star_rating": int(float(record.get("rating", 3))),
                                    "category": cat,
                                })
                        if len(records) >= per_cat:
                            break
                if records:
                    df_cat = pd.DataFrame(records)
                    print(f"  Strategy 2 (HF stream): Loaded {len(df_cat):,} reviews.")
                else:
                    df_cat = None
            except Exception as e:
                print(f"  Strategy 2 failed: {e}")
                df_cat = None

        # --- Strategy 3: UCSD mirror (JSONL.gz, streamed decompression) ---
        if df_cat is None:
            try:
                import gzip
                import requests as req
                from io import BytesIO
                url = ("https://datarepo.eng.ucsd.edu/mcauley_group/data/"
                       f"amazon_2023/raw/review_categories/{cat}.jsonl.gz")
                print(f"  Strategy 3: Streaming from UCSD mirror (gzip)...")
                records = []
                with req.get(url, stream=True, timeout=120) as resp:
                    resp.raise_for_status()
                    buf = BytesIO()
                    for chunk in resp.iter_content(chunk_size=1024*1024):
                        buf.write(chunk)
                        buf.seek(0)
                        try:
                            decompressor = gzip.GzipFile(fileobj=BytesIO(buf.getvalue()))
                            for line in decompressor:
                                record = json.loads(line)
                                text = str(record.get("text", "")).strip()
                                if len(text) > 10:
                                    records.append({
                                        "review_text": text,
                                        "star_rating": int(float(record.get("rating", 3))),
                                        "category": cat,
                                    })
                                if len(records) >= per_cat:
                                    break
                        except (gzip.BadGzipFile, EOFError):
                            pass  # incomplete chunk, wait for more
                        if len(records) >= per_cat:
                            break
                        buf.seek(0, 2)  # seek to end for next write
                if records:
                    df_cat = pd.DataFrame(records)
                    print(f"  Strategy 3 (UCSD gzip): Loaded {len(df_cat):,} reviews.")
                else:
                    df_cat = None
            except Exception as e:
                print(f"  Strategy 3 failed: {e}")
                df_cat = None

        # --- Strategy 4: Synthetic demo data ---
        if df_cat is None:
            print(f"  All download strategies failed. Using synthetic demo data.")
            print(f"  TIP: Run  pip install 'datasets<4'  to enable Strategy 1.")
            print(f"  WARNING: Synthetic data will produce non-meaningful results.")
            print(f"  WARNING: Ensure internet access for real Amazon review data.")
            df_cat = _make_demo_data(per_cat, cat)

        frames.append(df_cat)

    combined = pd.concat(frames, ignore_index=True)
    combined["star_rating"] = combined["star_rating"].astype(int)
    combined["sentiment_label"] = combined["star_rating"].map(
        {1: 0, 2: 0, 3: 1, 4: 2, 5: 2}
    )
    combined = combined.dropna(subset=["review_text"])
    combined = combined[combined["review_text"].str.strip().str.len() > 10]
    return combined.reset_index(drop=True)


def _make_demo_data(n: int, category: str) -> pd.DataFrame:
    """Synthetic data for pipeline validation when download is unavailable."""
    rng = np.random.RandomState(42)
    ratings = rng.choice([1,2,3,4,5], size=n, p=[0.08, 0.07, 0.10, 0.20, 0.55])
    templates = {
        5: ["Excellent product, works perfectly. Great value for money and fast shipping.",
            "Love this item! The quality exceeded my expectations completely.",
            "Best purchase I have made in years. Highly recommend to everyone.",
            "Amazing quality and performance. The build is solid and durable.",
            "Perfect for what I needed. Easy setup and great customer support."],
        4: ["Good product overall. Minor issues with packaging but works well.",
            "Solid purchase. Battery life could be better but camera is great.",
            "Happy with this buy. Performs as described with minor caveats.",
            "Nice product for the price. Shipping was fast and packaging decent."],
        3: ["Average product. Nothing special but does the job adequately.",
            "Mixed feelings. Good features but build quality is mediocre.",
            "It works as described but nothing impressive about the quality.",
            "Okay for the price point. Would not buy again at full retail."],
        2: ["Disappointed with this purchase. Quality is below average for price.",
            "Not worth the price. Screen resolution is poor and battery drains.",
            "Expected better quality. The materials feel cheap and flimsy.",
            "Product arrived damaged. Customer service was unhelpful about refund."],
        1: ["Terrible product. Broke after two days of normal use.",
            "Worst purchase ever. Does not work as advertised at all.",
            "Complete waste of money. Returned immediately after opening.",
            "Do not buy this. It is a scam product with fake positive reviews."],
    }
    texts = [rng.choice(templates[r]) for r in ratings]
    return pd.DataFrame({"review_text": texts, "star_rating": ratings, "category": category})


# --- Load ---
total_needed = config.train_size + config.val_size + config.test_size
df_reviews = load_amazon_reviews(config.amazon_categories, total_needed)
print(f"\nTotal loaded: {len(df_reviews):,}")
print(df_reviews["sentiment_label"].value_counts().sort_index().rename(
    {0: "Negative", 1: "Neutral", 2: "Positive"}
))

In [ ]:
# ============================================================================
# CELL 5: EDA + Class Weights
# ============================================================================

LABEL_NAMES = ["Negative", "Neutral", "Positive"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df_reviews["star_rating"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#d32f2f","#f57c00","#fbc02d","#7cb342","#388e3c"]
)
axes[0].set_title("Star Rating Distribution", fontweight="bold")
axes[0].set_xlabel("Stars"); axes[0].set_ylabel("Count")

df_reviews["sentiment_label"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1], color=["#d32f2f","#fbc02d","#388e3c"]
)
axes[1].set_title("Sentiment Distribution", fontweight="bold")
axes[1].set_xticklabels(LABEL_NAMES, rotation=0)

df_reviews["word_count"] = df_reviews["review_text"].str.split().str.len()
df_reviews["word_count"].clip(upper=500).hist(bins=50, ax=axes[2], color="#1565c0", alpha=0.7)
axes[2].set_title("Review Length (words)", fontweight="bold")
axes[2].axvline(df_reviews["word_count"].median(), color="red", ls="--",
                label=f"Median: {df_reviews['word_count'].median():.0f}")
axes[2].legend()
plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, "eda.png"), dpi=150)
plt.show()

# --- Compute class weights (used by loss functions later) ---
class_counts = df_reviews["sentiment_label"].value_counts().sort_index()
CLASS_WEIGHTS = torch.tensor(
    ((1.0 / class_counts) * len(df_reviews) / config.num_sentiment_classes).values,
    dtype=torch.float
).to(DEVICE)
print("\nClass weights (tensor on device):")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name}: {CLASS_WEIGHTS[i].item():.3f}")

In [ ]:
# ============================================================================
# CELL 6: Train/Val/Test Split
# ============================================================================

def create_splits(df, train_n, val_n, test_n, seed=42):
    """Stratified train/val/test split."""
    total = train_n + val_n + test_n
    if len(df) > total:
        df = df.sample(n=total, random_state=seed, replace=False)
    elif len(df) < total:
        print(f"Warning: only {len(df)} reviews available, adjusting split sizes.")
        ratio_val = val_n / total
        ratio_test = test_n / total
        test_n = int(len(df) * ratio_test)
        val_n = int(len(df) * ratio_val)
        train_n = len(df) - val_n - test_n

    tv, test = train_test_split(df, test_size=test_n, random_state=seed,
                                 stratify=df["sentiment_label"])
    train, val = train_test_split(tv, test_size=val_n, random_state=seed,
                                   stratify=tv["sentiment_label"])
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


df_train, df_val, df_test = create_splits(
    df_reviews, config.train_size, config.val_size, config.test_size
)

for name, sdf in [("train", df_train), ("val", df_val), ("test", df_test)]:
    sdf.to_csv(os.path.join(config.output_dir, f"{name}.csv"), index=False)
    print(f"{name:5s}: {len(sdf):>6,} | "
          f"Pos {(sdf['sentiment_label']==2).mean():.1%} | "
          f"Neu {(sdf['sentiment_label']==1).mean():.1%} | "
          f"Neg {(sdf['sentiment_label']==0).mean():.1%}")

In [ ]:
# ============================================================================
# CELL 6b: Load SemEval-2014 Task 4 for Aspect-Level Training & Evaluation
# ============================================================================
# SemEval-2014 provides gold-standard aspect term annotations (BIO tags)
# and aspect-level sentiment labels — required for ATE and ASC evaluation.

import xml.etree.ElementTree as ET
from io import BytesIO
import requests

BIO_TAG_MAP = {"B-ASP": 0, "I-ASP": 1, "O": 2}
BIO_TAG_NAMES = ["B-ASP", "I-ASP", "O"]

def _download_semeval(url):
    """Download SemEval XML from GitHub mirrors."""
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        return resp.content
    except Exception as e:
        print(f"  Download failed: {e}")
        return None

def _parse_semeval_xml(xml_bytes):
    """Parse SemEval-2014 Task 4 XML format into aspect records."""
    records = []
    root = ET.fromstring(xml_bytes)
    for sentence in root.iter("sentence"):
        text = sentence.find("text").text
        if text is None:
            continue
        aspects = []
        asp_terms = sentence.find("aspectTerms")
        if asp_terms is not None:
            for at in asp_terms.findall("aspectTerm"):
                term = at.get("term")
                polarity = at.get("polarity")
                start = int(at.get("from"))
                end = int(at.get("to"))
                if polarity in ("positive", "negative", "neutral"):
                    aspects.append({
                        "term": term, "polarity": polarity,
                        "start": start, "end": end
                    })
        if aspects:
            records.append({"text": text, "aspects": aspects})
    return records

def _create_bio_labels(text, aspects, tokenizer, max_len):
    """Create BIO tag sequence aligned to BERT tokenisation."""
    enc = tokenizer(text, max_length=max_len, padding="max_length",
                    truncation=True, return_offsets_mapping=True, return_tensors="pt")
    offsets = enc["offset_mapping"][0].tolist()  # list of (start, end) per token
    bio = [2] * max_len  # default O

    for asp in aspects:
        a_start, a_end = asp["start"], asp["end"]
        first = True
        for idx, (ts, te) in enumerate(offsets):
            if ts == 0 and te == 0:
                continue  # special token
            if ts >= a_start and te <= a_end:
                bio[idx] = 0 if first else 1  # B-ASP or I-ASP
                first = False
    return enc, bio

def _polarity_to_int(polarity):
    return {"negative": 0, "neutral": 1, "positive": 2}[polarity]

def load_semeval_data(tokenizer, max_len=128):
    """Load SemEval-2014 Laptop + Restaurant data with BIO labels."""
    # GitHub-hosted mirrors of SemEval-2014 Task 4 data
    urls = {
        "laptop_train": "https://raw.githubusercontent.com/songanyang/ABSA-PyTorch/master/datasets/semeval14/Laptops_Train.xml",
        "laptop_test": "https://raw.githubusercontent.com/songanyang/ABSA-PyTorch/master/datasets/semeval14/Laptops_Test_Gold.xml",
        "restaurant_train": "https://raw.githubusercontent.com/songanyang/ABSA-PyTorch/master/datasets/semeval14/Restaurants_Train.xml",
        "restaurant_test": "https://raw.githubusercontent.com/songanyang/ABSA-PyTorch/master/datasets/semeval14/Restaurants_Test_Gold.xml",
    }

    all_train, all_test = [], []
    for name, url in urls.items():
        print(f"  Downloading {name}...")
        xml = _download_semeval(url)
        if xml is None:
            continue
        records = _parse_semeval_xml(xml)
        target = all_train if "train" in name else all_test
        domain = "laptop" if "laptop" in name else "restaurant"
        for r in records:
            r["domain"] = domain
        target.extend(records)
        print(f"    Parsed {len(records)} sentences with aspects.")

    if not all_train:
        print("  WARNING: SemEval download failed. Using synthetic aspect data.")
        all_train, all_test = _make_synthetic_aspects()

    # Build tensors
    def _build_aspect_dataset(records):
        items = []
        for r in records:
            enc, bio = _create_bio_labels(r["text"], r["aspects"], tokenizer, max_len)
            for asp in r["aspects"]:
                items.append({
                    "input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "bio_labels": torch.tensor(bio, dtype=torch.long),
                    "asp_sentiment": torch.tensor(_polarity_to_int(asp["polarity"]), dtype=torch.long),
                    "text": r["text"],
                    "term": asp["term"],
                    "domain": r.get("domain", "unknown"),
                })
        return items

    train_items = _build_aspect_dataset(all_train)
    test_items = _build_aspect_dataset(all_test)
    print(f"\nSemEval aspect data: {len(train_items)} train, {len(test_items)} test")
    return train_items, test_items

def _make_synthetic_aspects():
    """Fallback synthetic data when SemEval is unavailable."""
    train = [
        {"text": "The battery life is great but the screen is terrible.",
         "aspects": [{"term":"battery life","polarity":"positive","start":4,"end":16},
                     {"term":"screen","polarity":"negative","start":34,"end":40}]},
        {"text": "Amazing camera quality and fast processor.",
         "aspects": [{"term":"camera quality","polarity":"positive","start":8,"end":22},
                     {"term":"processor","polarity":"positive","start":32,"end":41}]},
        {"text": "Terrible build quality, waste of money.",
         "aspects": [{"term":"build quality","polarity":"negative","start":9,"end":22}]},
    ] * 200  # repeat for minimum training data
    test = [
        {"text": "The keyboard is comfortable but speakers are awful.",
         "aspects": [{"term":"keyboard","polarity":"positive","start":4,"end":12},
                     {"term":"speakers","polarity":"negative","start":31,"end":39}]},
        {"text": "Excellent display with vibrant colours.",
         "aspects": [{"term":"display","polarity":"positive","start":10,"end":17}]},
    ] * 50
    return train, test


# --- Load SemEval data ---
# Initialize tokenizer early (also used in ReviewDataset later)
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained(config.bert_model_name)
print(f"Tokenizer loaded: {config.bert_model_name}")

print("Loading SemEval-2014 aspect data...")
semeval_train, semeval_test = load_semeval_data(tokenizer, config.max_seq_length)

class AspectDataset(Dataset):
    """Dataset for aspect-level training (ATE + ASC)."""
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        it = self.items[idx]
        return {
            "input_ids": it["input_ids"],
            "attention_mask": it["attention_mask"],
            "bio_labels": it["bio_labels"],
            "asp_sentiment": it["asp_sentiment"],
        }

aspect_train_ds = AspectDataset(semeval_train)
aspect_test_ds = AspectDataset(semeval_test)
print(f"Aspect datasets: Train={len(aspect_train_ds)}, Test={len(aspect_test_ds)}")


---
## 3. Model Architecture

| Head | Task | Input | Output | Loss |
|------|------|-------|--------|------|
| Sentence | Per-sentence sentiment | [CLS] | 3-class | Cross-entropy |
| Aspect Extract | BIO tagging | Token states | Tag/token | CRF NLL |
| Aspect Sentiment | Per-aspect polarity | ASP ⊕ CLS | 3-class | Cross-entropy |
| Document | Document sentiment | Attn over [CLS]s | 3-class | Cross-entropy |

BERT is **fine-tuned** end-to-end — no LSTM needed (Sun et al., 2019).

In [ ]:
# ============================================================================
# CELL 7: Document-Level Attention Aggregator
# ============================================================================
# Weights sentence [CLS] vectors by learned attention, aggregates into
# a single document vector. Based on Yang et al. (2016) HAN.

class DocumentAggregator(nn.Module):
    """Query-based attention over sentence [CLS] representations."""

    def __init__(self, hidden_dim=768, num_labels=3, dropout=0.3):
        super().__init__()
        self.query = nn.Parameter(torch.randn(hidden_dim))
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.scale = hidden_dim ** 0.5
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_labels),
        )

    def forward(self, sent_reprs, sent_mask, labels=None):
        keys = self.key_proj(sent_reprs)
        scores = torch.einsum("bsh,h->bs", keys, self.query) / self.scale
        scores = scores.masked_fill(~sent_mask.bool(), float("-inf"))
        attn = F.softmax(scores, dim=-1)
        doc = torch.einsum("bs,bsh->bh", attn, sent_reprs)
        logits = self.classifier(doc)
        out = {"logits": logits, "attention_weights": attn}
        if labels is not None:
            out["loss"] = F.cross_entropy(logits, labels, weight=CLASS_WEIGHTS)
        return out


print("DocumentAggregator defined.")

In [ ]:
# ============================================================================
# CELL 8: HMGS Model — Unified multi-task architecture
# ============================================================================

class HMGS(nn.Module):
    """Hierarchical Multi-Granularity Sentiment model.
    Shared BERT encoder with 4 task heads trained jointly."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.config = cfg
        self.bert = BertModel.from_pretrained(cfg.bert_model_name)
        h = self.bert.config.hidden_size  # 768

        # Head 1: Sentence sentiment
        self.sent_head = nn.Sequential(
            nn.Dropout(cfg.classifier_dropout),
            nn.Linear(h, h // 2), nn.GELU(),
            nn.Dropout(cfg.classifier_dropout),
            nn.Linear(h // 2, cfg.num_sentiment_classes),
        )

        # Head 2: Aspect extraction (BIO + CRF)
        self.ate_proj = nn.Linear(h, cfg.num_aspect_bio_tags)
        self.crf = CRF(cfg.num_aspect_bio_tags, batch_first=True)

        # Head 3: Aspect sentiment
        self.asc_head = nn.Sequential(
            nn.Dropout(cfg.classifier_dropout),
            nn.Linear(h * 2, h // 2), nn.GELU(),
            nn.Dropout(cfg.classifier_dropout),
            nn.Linear(h // 2, cfg.num_sentiment_classes),
        )

        # Head 4: Document aggregation
        self.doc_agg = DocumentAggregator(h, cfg.num_sentiment_classes, cfg.classifier_dropout)

    def encode(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state  # (B, L, H)

    def forward_sent(self, cls_repr, labels=None):
        logits = self.sent_head(cls_repr)
        r = {"sent_logits": logits}
        if labels is not None:
            r["sent_loss"] = F.cross_entropy(logits, labels, weight=CLASS_WEIGHTS)
        return r

    def forward_ate(self, token_states, attention_mask, bio_labels=None):
        emissions = self.ate_proj(token_states)
        mask = attention_mask.bool()
        r = {"tags": self.crf.decode(emissions, mask=mask)}
        if bio_labels is not None:
            r["ate_loss"] = -self.crf(emissions, bio_labels, mask=mask, reduction="mean")
        return r

    def forward_asc(self, aspect_repr, cls_repr, labels=None):
        logits = self.asc_head(torch.cat([aspect_repr, cls_repr], dim=-1))
        r = {"asc_logits": logits}
        if labels is not None:
            r["asc_loss"] = F.cross_entropy(logits, labels, weight=CLASS_WEIGHTS)
        return r

    def forward_doc(self, sent_cls_batch, sent_mask, labels=None):
        return self.doc_agg(sent_cls_batch, sent_mask, labels)


# --- Instantiate ---
model = HMGS(config).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,} total, {n_train:,} trainable")

---
## 4. Training Protocol

- **Staged training:** Phase 1 trains document + sentence heads on Amazon reviews; Phase 2 fine-tunes ATE + ASC heads on SemEval-2014 gold annotations.
- **Sentence labels:** Distant supervision from document label (noisy but standard)
- **Discriminative LR:** 2e-5 for BERT, 1e-3 for task heads
- **Gradient accumulation:** Effective batch = 32
- **Mixed precision (FP16)** for memory efficiency
- **Early stopping** on validation macro-F1

In [ ]:
# ============================================================================
# CELL 9: Dataset
# ============================================================================

class ReviewDataset(Dataset):
    """Segments reviews into sentences, tokenises each for BERT."""

    def __init__(self, texts, labels, tokenizer, max_len=128, max_sents=10):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.max_len = max_len
        self.max_sents = max_sents

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        sents = sent_tokenize(text)[:self.max_sents]
        n = len(sents)

        ids_all, mask_all = [], []
        for s in sents:
            enc = self.tok(s, max_length=self.max_len, padding="max_length",
                           truncation=True, return_tensors="pt")
            ids_all.append(enc["input_ids"].squeeze(0))
            mask_all.append(enc["attention_mask"].squeeze(0))

        pad = self.max_sents - n
        if pad > 0:
            ids_all.append(torch.zeros(pad, self.max_len, dtype=torch.long))
            mask_all.append(torch.zeros(pad, self.max_len, dtype=torch.long))

        input_ids = torch.cat([t.unsqueeze(0) if t.dim()==1 else t for t in ids_all], dim=0)
        attn_mask = torch.cat([t.unsqueeze(0) if t.dim()==1 else t for t in mask_all], dim=0)
        sent_mask = torch.zeros(self.max_sents, dtype=torch.long)
        sent_mask[:n] = 1

        return {
            "input_ids": input_ids,          # (max_sents, max_len)
            "attention_mask": attn_mask,      # (max_sents, max_len)
            "sent_mask": sent_mask,           # (max_sents,)
            "doc_label": torch.tensor(label, dtype=torch.long),
            "sent_labels": torch.full((self.max_sents,), label, dtype=torch.long),
        }


# tokenizer already initialized in CELL 6b; re-use it
if 'tokenizer' not in dir():
    tokenizer = BertTokenizerFast.from_pretrained(config.bert_model_name)

train_ds = ReviewDataset(df_train["review_text"].tolist(), df_train["sentiment_label"].tolist(),
                          tokenizer, config.max_seq_length, config.max_doc_sentences)
val_ds = ReviewDataset(df_val["review_text"].tolist(), df_val["sentiment_label"].tolist(),
                        tokenizer, config.max_seq_length, config.max_doc_sentences)
test_ds = ReviewDataset(df_test["review_text"].tolist(), df_test["sentiment_label"].tolist(),
                         tokenizer, config.max_seq_length, config.max_doc_sentences)

print(f"Datasets — Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

# Quick sanity check
sample = train_ds[0]
print(f"Sample shape: input_ids={sample['input_ids'].shape}, sent_mask={sample['sent_mask'].shape}")


In [ ]:
# ============================================================================
# CELL 10: Training Engine
# ============================================================================

class Trainer:
    def __init__(self, model, cfg, train_ds, val_ds):
        self.model = model
        self.cfg = cfg
        self.train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                                       shuffle=True, num_workers=0, pin_memory=True)
        self.val_loader = DataLoader(val_ds, batch_size=cfg.batch_size,
                                     shuffle=False, num_workers=0, pin_memory=True)

        bert_p = list(model.bert.parameters())
        head_p = [p for n, p in model.named_parameters() if not n.startswith("bert.")]

        self.optim = torch.optim.AdamW([
            {"params": bert_p, "lr": cfg.learning_rate_bert},
            {"params": head_p, "lr": cfg.learning_rate_heads},
        ], weight_decay=cfg.weight_decay)

        steps = (len(self.train_loader) // cfg.gradient_accumulation_steps) * cfg.max_epochs
        self.sched = get_linear_schedule_with_warmup(
            self.optim, int(steps * cfg.warmup_ratio), steps
        )
        self.scaler = GradScaler(enabled=(DEVICE.type == "cuda"))
        self.best_f1 = 0.0
        self.patience = 0

    def _encode_batch(self, ids, mask, smask):
        """Encode only real sentences through BERT, return [CLS] per sentence."""
        B, S, L = ids.shape
        flat_ids = ids.view(B * S, L)
        flat_mask = mask.view(B * S, L)
        real = smask.view(B * S).bool()

        all_cls = torch.zeros(B * S, self.cfg.hidden_dim, device=DEVICE)
        if real.any():
            out = self.model.bert(input_ids=flat_ids[real], attention_mask=flat_mask[real])
            all_cls[real] = out.last_hidden_state[:, 0, :]
        return all_cls.view(B, S, -1)

    def _train_epoch(self):
        self.model.train()
        total_loss, nb = 0.0, 0
        self.optim.zero_grad()

        for step, batch in enumerate(self.train_loader):
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            smask = batch["sent_mask"].to(DEVICE)
            doc_lab = batch["doc_label"].to(DEVICE)
            sent_lab = batch["sent_labels"].to(DEVICE)

            with amp_autocast():
                sent_cls = self._encode_batch(ids, mask, smask)

                # Sentence loss (on real sentences only)
                s_res = self.model.forward_sent(
                    sent_cls[smask.bool()], sent_lab[smask.bool()]
                )
                # Document loss
                d_res = self.model.forward_doc(sent_cls, smask, doc_lab)

                loss = (
                    self.cfg.lambda_doc * d_res.get("loss", 0.0)
                    + self.cfg.lambda_sent * s_res.get("sent_loss", 0.0)
                ) / self.cfg.gradient_accumulation_steps

            self.scaler.scale(loss).backward()

            if (step + 1) % self.cfg.gradient_accumulation_steps == 0:
                self.scaler.unscale_(self.optim)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.optim)
                self.scaler.update()
                self.sched.step()
                self.optim.zero_grad()

            total_loss += loss.item() * self.cfg.gradient_accumulation_steps
            nb += 1

        return total_loss / max(nb, 1)

    @torch.no_grad()
    def _evaluate(self):
        self.model.eval()
        preds, labels = [], []
        total_loss, nb = 0.0, 0

        for batch in self.val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            smask = batch["sent_mask"].to(DEVICE)
            doc_lab = batch["doc_label"].to(DEVICE)

            sent_cls = self._encode_batch(ids, mask, smask)
            d_res = self.model.forward_doc(sent_cls, smask, doc_lab)

            total_loss += d_res["loss"].item()
            preds.extend(d_res["logits"].argmax(-1).cpu().numpy())
            labels.extend(doc_lab.cpu().numpy())
            nb += 1

        yt, yp = np.array(labels), np.array(preds)
        return {
            "val_loss": total_loss / max(nb, 1),
            "val_acc": accuracy_score(yt, yp),
            "val_f1": f1_score(yt, yp, average="macro"),
        }

    def train(self, seed):
        set_seed(seed)
        print(f"\n{'='*50} Seed {seed} {'='*50}")
        self.best_f1 = 0.0
        self.patience = 0
        history = []

        for epoch in range(self.cfg.max_epochs):
            t_loss = self._train_epoch()
            v = self._evaluate()
            history.append({"epoch": epoch+1, "train_loss": t_loss, **v})

            print(f"  Ep {epoch+1}/{self.cfg.max_epochs} | "
                  f"Loss {t_loss:.4f}/{v['val_loss']:.4f} | "
                  f"F1 {v['val_f1']:.4f} | Acc {v['val_acc']:.4f}")

            if v["val_f1"] > self.best_f1:
                self.best_f1 = v["val_f1"]
                self.patience = 0
                torch.save(self.model.state_dict(),
                           os.path.join(self.cfg.checkpoint_dir, f"best_s{seed}.pt"))
            else:
                self.patience += 1
                if self.patience >= self.cfg.early_stopping_patience:
                    print(f"  Early stop at epoch {epoch+1}")
                    break
        return history


print("Trainer defined.")

---
## 5. Evaluation Framework

5-seed evaluation with mean ± std reporting and bootstrap significance testing.

In [ ]:
# ============================================================================
# CELL 11: Evaluation functions
# ============================================================================

@torch.no_grad()
def evaluate_test(model, test_ds, cfg):
    """Evaluate model on test set, return all metrics."""
    model.eval()
    loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)
    preds, labels = [], []

    for batch in loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        smask = batch["sent_mask"].to(DEVICE)
        doc_lab = batch["doc_label"].to(DEVICE)
        B, S, L = ids.shape

        flat_ids, flat_mask = ids.view(B*S, L), mask.view(B*S, L)
        real = smask.view(B*S).bool()
        all_cls = torch.zeros(B*S, cfg.hidden_dim, device=DEVICE)
        if real.any():
            out = model.bert(input_ids=flat_ids[real], attention_mask=flat_mask[real])
            all_cls[real] = out.last_hidden_state[:, 0, :]

        d_res = model.forward_doc(all_cls.view(B, S, -1), smask)
        preds.extend(d_res["logits"].argmax(-1).cpu().numpy())
        labels.extend(doc_lab.cpu().numpy())

    yt, yp = np.array(labels), np.array(preds)
    return {
        "accuracy": accuracy_score(yt, yp),
        "f1_macro": f1_score(yt, yp, average="macro"),
        "precision": precision_score(yt, yp, average="macro"),
        "recall": recall_score(yt, yp, average="macro"),
        "kappa": cohen_kappa_score(yt, yp),
    }, yt, yp


def run_experiment(cfg, train_ds, val_ds, test_ds):
    """Full multi-seed experiment."""
    results = {m: [] for m in ["accuracy","f1_macro","precision","recall","kappa"]}
    all_histories = []

    for seed in cfg.seeds:
        mdl = HMGS(cfg).to(DEVICE)
        trainer = Trainer(mdl, cfg, train_ds, val_ds)
        hist = trainer.train(seed)
        all_histories.append(hist)

        # Load best checkpoint
        mdl.load_state_dict(torch.load(
            os.path.join(cfg.checkpoint_dir, f"best_s{seed}.pt"),
            map_location=DEVICE
        ))
        metrics, _, _ = evaluate_test(mdl, test_ds, cfg)
        for k, v in metrics.items():
            results[k].append(v)
        print(f"  Test: {metrics}")

        del mdl, trainer
        clear_gpu_cache()

    results = {k: np.array(v) for k, v in results.items()}

    print(f"\n{'='*60}")
    print(f"RESULTS ({len(cfg.seeds)} seeds)")
    print(f"{'='*60}")
    for m, v in results.items():
        print(f"  {m:15s}: {v.mean():.4f} ± {v.std():.4f}")
    print(f"{'='*60}")

    return results, all_histories


print("Evaluation functions defined.")

---
## 6. Baselines & Ablation

| Model | Purpose |
|-------|---------|
| TF-IDF + Logistic Regression | Traditional ML lower bound |
| BERT + BiLSTM + Attention | Original thesis architecture (ablation) |

In [ ]:
# ============================================================================
# CELL 12: TF-IDF Baseline
# ============================================================================

def tfidf_baseline(train_texts, train_labels, test_texts, test_labels):
    pipe = SkPipeline([
        ("tfidf", TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced",
                                    solver="lbfgs", multi_class="multinomial")),
    ])
    pipe.fit(train_texts, train_labels)
    yp = pipe.predict(test_texts)
    return {
        "accuracy": accuracy_score(test_labels, yp),
        "f1_macro": f1_score(test_labels, yp, average="macro"),
        "precision": precision_score(test_labels, yp, average="macro"),
        "recall": recall_score(test_labels, yp, average="macro"),
        "kappa": cohen_kappa_score(test_labels, yp),
    }, test_labels, yp

In [ ]:
# ============================================================================
# CELL 13: BERT + BiLSTM + Attention (original thesis architecture)
# ============================================================================

class BertLstmAttention(nn.Module):
    """BERT → BiLSTM → Attention → Classification."""

    def __init__(self, bert_name="bert-base-uncased", lstm_h=256,
                 num_labels=3, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_name)
        h = self.bert.config.hidden_size
        self.lstm = nn.LSTM(h, lstm_h, batch_first=True, bidirectional=True)
        dim = lstm_h * 2
        self.attn_w = nn.Linear(dim, 1)
        self.clf = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(dim, dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim//2, num_labels),
        )

    def forward(self, input_ids, attention_mask, labels=None):
        tok = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.lstm(tok)
        sc = self.attn_w(lstm_out).squeeze(-1)
        sc = sc.masked_fill(~attention_mask.bool(), float("-inf"))
        w = F.softmax(sc, dim=-1)
        ctx = torch.bmm(w.unsqueeze(1), lstm_out).squeeze(1)
        logits = self.clf(ctx)
        r = {"logits": logits, "attention_weights": w}
        if labels is not None:
            r["loss"] = F.cross_entropy(logits, labels, weight=CLASS_WEIGHTS)
        return r


print("Baselines defined.")

In [ ]:
# ============================================================================
# CELL 13b: Plain BERT + Linear Head Baseline
# ============================================================================
# Standard BERT fine-tuning baseline: [CLS] -> Linear -> 3-class.
# This is the strongest single-model baseline and the most common approach.

class BertLinearBaseline(nn.Module):
    """BERT [CLS] -> Dropout -> Linear -> 3-class sentiment."""
    def __init__(self, bert_name="bert-base-uncased", num_labels=3, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_name)
        h = self.bert.config.hidden_size
        self.clf = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(h, num_labels),
        )

    def forward(self, input_ids, attention_mask, labels=None):
        cls = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        logits = self.clf(cls)
        r = {"logits": logits}
        if labels is not None:
            r["loss"] = F.cross_entropy(logits, labels, weight=CLASS_WEIGHTS)
        return r

print("BertLinearBaseline defined.")

---
## 7. Visualisation

In [ ]:
# ============================================================================
# CELL 14: Visualisation utilities
# ============================================================================

def plot_confusion(yt, yp, names=LABEL_NAMES, title="Confusion Matrix", save=None):
    cm = confusion_matrix(yt, yp)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=names, yticklabels=names, ax=axes[0])
    axes[0].set_title(f"{title} (Counts)", fontweight="bold")
    axes[0].set_ylabel("True"); axes[0].set_xlabel("Predicted")
    sns.heatmap(cm_n, annot=True, fmt=".1%", cmap="Blues",
                xticklabels=names, yticklabels=names, ax=axes[1])
    axes[1].set_title(f"{title} (Normalised)", fontweight="bold")
    axes[1].set_ylabel("True"); axes[1].set_xlabel("Predicted")
    plt.tight_layout()
    if save: plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()


def plot_curves(history, save=None):
    eps = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(eps, [h["train_loss"] for h in history], "b-o", label="Train")
    axes[0].plot(eps, [h["val_loss"] for h in history], "r-o", label="Val")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss", fontweight="bold"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(eps, [h["val_f1"] for h in history], "g-o", label="Macro F1")
    axes[1].plot(eps, [h["val_acc"] for h in history], "m-o", label="Accuracy")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
    axes[1].set_title("Metrics", fontweight="bold"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    if save: plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()


def plot_doc_attention(model, text, tokenizer, cfg, save=None):
    """Visualise per-sentence attention weights for a review."""
    model.eval()
    sents = sent_tokenize(text)[:cfg.max_doc_sentences]
    cls_list = []
    for s in sents:
        enc = tokenizer(s, max_length=cfg.max_seq_length, padding="max_length",
                       truncation=True, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            cls_list.append(model.bert(**enc).last_hidden_state[:, 0, :])
    nr = len(cls_list)
    while len(cls_list) < cfg.max_doc_sentences:
        cls_list.append(torch.zeros(1, cfg.hidden_dim, device=DEVICE))
    sc = torch.cat(cls_list, dim=0).unsqueeze(0)
    sm = torch.zeros(1, cfg.max_doc_sentences, dtype=torch.long, device=DEVICE)
    sm[0, :nr] = 1
    with torch.no_grad():
        res = model.forward_doc(sc, sm)
    w = res["attention_weights"][0, :nr].cpu().numpy()
    pred = LABEL_NAMES[res["logits"][0].argmax().item()]
    labs = [s[:80]+("..." if len(s)>80 else "") for s in sents]
    fig, ax = plt.subplots(figsize=(12, max(3, nr*0.6)))
    ax.barh(range(nr), w, color=plt.cm.RdYlGn(w/w.max()))
    ax.set_yticks(range(nr)); ax.set_yticklabels(labs, fontsize=9)
    ax.set_xlabel("Attention Weight")
    ax.set_title(f"Predicted: {pred}", fontweight="bold")
    ax.invert_yaxis(); plt.tight_layout()
    if save: plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()


print("Visualisation functions defined.")

---
## 8. Execution

Run all cells above first. Then execute below to train and evaluate.

In [ ]:
# ============================================================================
# CELL 15: Run TF-IDF Baseline
# ============================================================================

print("Running TF-IDF baseline...")
tfidf_metrics, tfidf_yt, tfidf_yp = tfidf_baseline(
    df_train["review_text"].tolist(), df_train["sentiment_label"].tolist(),
    df_test["review_text"].tolist(), df_test["sentiment_label"].tolist(),
)
print(f"TF-IDF Results: {tfidf_metrics}")
print("\nClassification Report (TF-IDF):")
print(classification_report(tfidf_yt, tfidf_yp, target_names=LABEL_NAMES))
plot_confusion(tfidf_yt, tfidf_yp, title="TF-IDF Baseline",
               save=os.path.join(config.output_dir, "cm_tfidf.png"))

In [ ]:
# ============================================================================
# CELL 16: Run HMGS Model (single seed for quick test, uncomment loop for full)
# ============================================================================

# --- Single seed quick run ---
model = HMGS(config).to(DEVICE)
trainer = Trainer(model, config, train_ds, val_ds)
history = trainer.train(seed=42)

# Load best checkpoint
model.load_state_dict(torch.load(
    os.path.join(config.checkpoint_dir, "best_s42.pt"), map_location=DEVICE
))

# Free trainer memory
del trainer
clear_gpu_cache()

# Test evaluation
hmgs_metrics, hmgs_yt, hmgs_yp = evaluate_test(model, test_ds, config)
print(f"\nHMGS Results: {hmgs_metrics}")

# Plots
plot_curves(history, save=os.path.join(config.output_dir, "curves_hmgs.png"))
plot_confusion(hmgs_yt, hmgs_yp, title="HMGS Model",
               save=os.path.join(config.output_dir, "cm_hmgs.png"))

# Classification report
print("\nClassification Report:")
print(classification_report(hmgs_yt, hmgs_yp, target_names=LABEL_NAMES))

In [ ]:
# ============================================================================
# CELL 16b: Aspect-Level Fine-Tuning (ATE + ASC) on SemEval-2014
# ============================================================================
# Fine-tunes the HMGS model's ATE (CRF) and ASC heads on SemEval gold data.
# Uses the BERT encoder from the document-trained HMGS model (transfer learning).

print("Fine-tuning HMGS aspect heads on SemEval-2014 data...")

# Freeze BERT, only train aspect heads for efficiency
for p in model.bert.parameters():
    p.requires_grad = False
for p in model.ate_proj.parameters():
    p.requires_grad = True
for p in model.crf.parameters():
    p.requires_grad = True
for p in model.asc_head.parameters():
    p.requires_grad = True

asp_optim = torch.optim.AdamW([
    {"params": model.ate_proj.parameters(), "lr": 1e-3},
    {"params": model.crf.parameters(), "lr": 1e-3},
    {"params": model.asc_head.parameters(), "lr": 1e-3},
], weight_decay=0.01)

asp_train_loader = DataLoader(aspect_train_ds, batch_size=32, shuffle=True)
asp_test_loader = DataLoader(aspect_test_ds, batch_size=32, shuffle=False)

asp_epochs = 10
best_ate_f1 = 0.0

for epoch in range(asp_epochs):
    model.train()
    total_ate_loss, total_asc_loss, nb = 0.0, 0.0, 0

    for batch in asp_train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        bio = batch["bio_labels"].to(DEVICE)
        asc_lab = batch["asp_sentiment"].to(DEVICE)

        # Get token embeddings from frozen BERT
        with torch.no_grad():
            token_states = model.bert(input_ids=ids, attention_mask=mask).last_hidden_state

        # ATE loss (CRF)
        ate_res = model.forward_ate(token_states, mask, bio)
        ate_loss = ate_res["ate_loss"]

        # ASC loss — use [CLS] + mean of B/I-ASP tokens as aspect repr
        cls_repr = token_states[:, 0, :]  # (B, H)
        # Compute aspect repr as mean of tokens tagged as B-ASP or I-ASP
        bio_mask = (bio == 0) | (bio == 1)  # B-ASP or I-ASP
        # If no aspect tokens predicted, use CLS as fallback
        asp_repr = torch.zeros_like(cls_repr)
        for b in range(ids.shape[0]):
            asp_tokens = token_states[b][bio_mask[b]]
            if len(asp_tokens) > 0:
                asp_repr[b] = asp_tokens.mean(dim=0)
            else:
                asp_repr[b] = cls_repr[b]

        asc_res = model.forward_asc(asp_repr, cls_repr, asc_lab)
        asc_loss = asc_res["asc_loss"]

        loss = ate_loss + asc_loss
        asp_optim.zero_grad()
        loss.backward()
        asp_optim.step()

        total_ate_loss += ate_loss.item()
        total_asc_loss += asc_loss.item()
        nb += 1

    # Quick validation
    model.eval()
    ate_preds_all, ate_labels_all = [], []
    asc_preds_all, asc_labels_all = [], []
    with torch.no_grad():
        for batch in asp_test_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            bio = batch["bio_labels"].to(DEVICE)
            asc_lab = batch["asp_sentiment"].to(DEVICE)

            token_states = model.bert(input_ids=ids, attention_mask=mask).last_hidden_state
            ate_res = model.forward_ate(token_states, mask)

            # Flatten BIO predictions vs labels (only on real tokens)
            for b in range(ids.shape[0]):
                real_len = mask[b].sum().item()
                pred_tags = ate_res["tags"][b][:real_len]
                true_tags = bio[b][:real_len].cpu().tolist()
                ate_preds_all.extend(pred_tags)
                ate_labels_all.extend(true_tags)

            # ASC — use CRF-predicted tags (not gold) for realistic evaluation
            cls_repr = token_states[:, 0, :]
            pred_bio_v = torch.full_like(bio, 2)  # default O
            for b in range(ids.shape[0]):
                rl = mask[b].sum().item()
                pt = ate_res["tags"][b][:rl]
                pred_bio_v[b, :rl] = torch.tensor(pt, device=DEVICE)
            bio_mask_val = (pred_bio_v == 0) | (pred_bio_v == 1)
            asp_repr = torch.zeros_like(cls_repr)
            for b in range(ids.shape[0]):
                asp_tokens = token_states[b][bio_mask_val[b]]
                if len(asp_tokens) > 0:
                    asp_repr[b] = asp_tokens.mean(dim=0)
                else:
                    asp_repr[b] = cls_repr[b]
            asc_res = model.forward_asc(asp_repr, cls_repr)
            asc_preds_all.extend(asc_res["asc_logits"].argmax(-1).cpu().numpy())
            asc_labels_all.extend(asc_lab.cpu().numpy())

    ate_f1 = f1_score(ate_labels_all, ate_preds_all, average="macro")
    asc_f1 = f1_score(asc_labels_all, asc_preds_all, average="macro")
    asc_acc = accuracy_score(asc_labels_all, asc_preds_all)

    print(f"  Ep {epoch+1}/{asp_epochs} | ATE-loss {total_ate_loss/nb:.4f} | "
          f"ASC-loss {total_asc_loss/nb:.4f} | ATE-F1 {ate_f1:.4f} | "
          f"ASC-F1 {asc_f1:.4f} | ASC-Acc {asc_acc:.4f}")

    if ate_f1 > best_ate_f1:
        best_ate_f1 = ate_f1
        torch.save(model.state_dict(), os.path.join(config.checkpoint_dir, "best_aspect.pt"))

# Restore best and unfreeze BERT
model.load_state_dict(torch.load(os.path.join(config.checkpoint_dir, "best_aspect.pt"),
                                  map_location=DEVICE))
for p in model.bert.parameters():
    p.requires_grad = True

print(f"\nAspect fine-tuning complete. Best ATE-F1: {best_ate_f1:.4f}")

In [ ]:
# ============================================================================
# CELL 16c: Run BERT+BiLSTM+Attention Ablation (Original Thesis Architecture)
# ============================================================================
# This trains the original thesis architecture for direct comparison.
# Uses sentence-level (flat) approach — encodes full review as single sequence.

class FlatReviewDataset(Dataset):
    """Flat tokenisation — full review as one sequence (for BERT+BiLSTM)."""
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tok(str(self.texts[idx]), max_length=self.max_len,
                       padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Build flat datasets
flat_train = FlatReviewDataset(df_train["review_text"].tolist(),
                                df_train["sentiment_label"].tolist(), tokenizer, 256)
flat_val = FlatReviewDataset(df_val["review_text"].tolist(),
                              df_val["sentiment_label"].tolist(), tokenizer, 256)
flat_test = FlatReviewDataset(df_test["review_text"].tolist(),
                               df_test["sentiment_label"].tolist(), tokenizer, 256)

# Train BERT+BiLSTM+Attention
set_seed(42)
ablation_model = BertLstmAttention().to(DEVICE)
ab_bert_p = list(ablation_model.bert.parameters())
ab_head_p = [p for n, p in ablation_model.named_parameters() if not n.startswith("bert.")]

ab_optim = torch.optim.AdamW([
    {"params": ab_bert_p, "lr": config.learning_rate_bert},
    {"params": ab_head_p, "lr": config.learning_rate_heads},
], weight_decay=config.weight_decay)

ab_train_loader = DataLoader(flat_train, batch_size=config.batch_size, shuffle=True)
ab_val_loader = DataLoader(flat_val, batch_size=config.batch_size, shuffle=False)
ab_test_loader = DataLoader(flat_test, batch_size=config.batch_size, shuffle=False)

ab_steps = (len(ab_train_loader) // config.gradient_accumulation_steps) * config.max_epochs
ab_sched = get_linear_schedule_with_warmup(ab_optim, int(ab_steps * config.warmup_ratio), ab_steps)
ab_scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

print("Training BERT+BiLSTM+Attention (ablation)...")
ab_best_f1, ab_patience = 0.0, 0
ab_history = []

for epoch in range(config.max_epochs):
    ablation_model.train()
    t_loss, nb = 0.0, 0
    ab_optim.zero_grad()
    for step, batch in enumerate(ab_train_loader):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labs = batch["labels"].to(DEVICE)
        with amp_autocast():
            res = ablation_model(ids, mask, labs)
            loss = res["loss"] / config.gradient_accumulation_steps
        ab_scaler.scale(loss).backward()
        if (step + 1) % config.gradient_accumulation_steps == 0:
            ab_scaler.unscale_(ab_optim)
            torch.nn.utils.clip_grad_norm_(ablation_model.parameters(), 1.0)
            ab_scaler.step(ab_optim)
            ab_scaler.update()
            ab_sched.step()
            ab_optim.zero_grad()
        t_loss += loss.item() * config.gradient_accumulation_steps
        nb += 1

    # Validation
    ablation_model.eval()
    vp, vl = [], []
    v_loss_tot, vnb = 0.0, 0
    with torch.no_grad():
        for batch in ab_val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labs = batch["labels"].to(DEVICE)
            res = ablation_model(ids, mask, labs)
            v_loss_tot += res["loss"].item()
            vp.extend(res["logits"].argmax(-1).cpu().numpy())
            vl.extend(labs.cpu().numpy())
            vnb += 1

    vf1 = f1_score(vl, vp, average="macro")
    vacc = accuracy_score(vl, vp)
    ab_history.append({"epoch": epoch+1, "train_loss": t_loss/max(nb,1),
                       "val_loss": v_loss_tot/max(vnb,1), "val_f1": vf1, "val_acc": vacc})
    print(f"  Ep {epoch+1}/{config.max_epochs} | Loss {t_loss/max(nb,1):.4f}/{v_loss_tot/max(vnb,1):.4f} | "
          f"F1 {vf1:.4f} | Acc {vacc:.4f}")

    if vf1 > ab_best_f1:
        ab_best_f1 = vf1
        ab_patience = 0
        torch.save(ablation_model.state_dict(), os.path.join(config.checkpoint_dir, "best_ablation.pt"))
    else:
        ab_patience += 1
        if ab_patience >= config.early_stopping_patience:
            print(f"  Early stop at epoch {epoch+1}")
            break

# Load best and evaluate
ablation_model.load_state_dict(torch.load(os.path.join(config.checkpoint_dir, "best_ablation.pt"),
                                           map_location=DEVICE))
ablation_model.eval()
ab_preds, ab_labels = [], []
with torch.no_grad():
    for batch in ab_test_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labs = batch["labels"].to(DEVICE)
        res = ablation_model(ids, mask)
        ab_preds.extend(res["logits"].argmax(-1).cpu().numpy())
        ab_labels.extend(labs.cpu().numpy())

ab_yt, ab_yp = np.array(ab_labels), np.array(ab_preds)
ablation_metrics = {
    "accuracy": accuracy_score(ab_yt, ab_yp),
    "f1_macro": f1_score(ab_yt, ab_yp, average="macro"),
    "precision": precision_score(ab_yt, ab_yp, average="macro"),
    "recall": recall_score(ab_yt, ab_yp, average="macro"),
    "kappa": cohen_kappa_score(ab_yt, ab_yp),
}
print(f"\nBERT+BiLSTM+Attn Results: {ablation_metrics}")

# Plots
print("\nClassification Report (BERT+BiLSTM+Attn):")
print(classification_report(ab_yt, ab_yp, target_names=LABEL_NAMES))
plot_curves(ab_history, save=os.path.join(config.output_dir, "curves_ablation.png"))
plot_confusion(ab_yt, ab_yp, title="BERT+BiLSTM+Attn (Ablation)",
               save=os.path.join(config.output_dir, "cm_ablation.png"))

# Free GPU memory
del ablation_model
clear_gpu_cache()

In [ ]:
# ============================================================================
# CELL 16d: Run BERT+Linear Baseline
# ============================================================================

set_seed(42)
bert_linear_model = BertLinearBaseline().to(DEVICE)
bl_bert_p = list(bert_linear_model.bert.parameters())
bl_head_p = [p for n, p in bert_linear_model.named_parameters() if not n.startswith("bert.")]

bl_optim = torch.optim.AdamW([
    {"params": bl_bert_p, "lr": config.learning_rate_bert},
    {"params": bl_head_p, "lr": config.learning_rate_heads},
], weight_decay=config.weight_decay)

# Reuse flat datasets from ablation cell
bl_train_loader = DataLoader(flat_train, batch_size=config.batch_size, shuffle=True)
bl_val_loader = DataLoader(flat_val, batch_size=config.batch_size, shuffle=False)
bl_test_loader = DataLoader(flat_test, batch_size=config.batch_size, shuffle=False)

bl_steps = (len(bl_train_loader) // config.gradient_accumulation_steps) * config.max_epochs
bl_sched = get_linear_schedule_with_warmup(bl_optim, int(bl_steps * config.warmup_ratio), bl_steps)
bl_scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

print("Training BERT+Linear baseline...")
bl_best_f1, bl_patience = 0.0, 0
bl_history = []

for epoch in range(config.max_epochs):
    bert_linear_model.train()
    t_loss, nb = 0.0, 0
    bl_optim.zero_grad()
    for step, batch in enumerate(bl_train_loader):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labs = batch["labels"].to(DEVICE)
        with amp_autocast():
            res = bert_linear_model(ids, mask, labs)
            loss = res["loss"] / config.gradient_accumulation_steps
        bl_scaler.scale(loss).backward()
        if (step + 1) % config.gradient_accumulation_steps == 0:
            bl_scaler.unscale_(bl_optim)
            torch.nn.utils.clip_grad_norm_(bert_linear_model.parameters(), 1.0)
            bl_scaler.step(bl_optim)
            bl_scaler.update()
            bl_sched.step()
            bl_optim.zero_grad()
        t_loss += loss.item() * config.gradient_accumulation_steps
        nb += 1

    bert_linear_model.eval()
    vp, vl = [], []
    v_loss_tot, vnb = 0.0, 0
    with torch.no_grad():
        for batch in bl_val_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labs = batch["labels"].to(DEVICE)
            res = bert_linear_model(ids, mask, labs)
            v_loss_tot += res["loss"].item()
            vp.extend(res["logits"].argmax(-1).cpu().numpy())
            vl.extend(labs.cpu().numpy())
            vnb += 1

    vf1 = f1_score(vl, vp, average="macro")
    vacc = accuracy_score(vl, vp)
    bl_history.append({"epoch": epoch+1, "train_loss": t_loss/max(nb,1),
                       "val_loss": v_loss_tot/max(vnb,1), "val_f1": vf1, "val_acc": vacc})
    print(f"  Ep {epoch+1}/{config.max_epochs} | Loss {t_loss/max(nb,1):.4f}/{v_loss_tot/max(vnb,1):.4f} | "
          f"F1 {vf1:.4f} | Acc {vacc:.4f}")

    if vf1 > bl_best_f1:
        bl_best_f1 = vf1
        bl_patience = 0
        torch.save(bert_linear_model.state_dict(), os.path.join(config.checkpoint_dir, "best_bert_linear.pt"))
    else:
        bl_patience += 1
        if bl_patience >= config.early_stopping_patience:
            print(f"  Early stop at epoch {epoch+1}")
            break

# Evaluate on test
bert_linear_model.load_state_dict(torch.load(os.path.join(config.checkpoint_dir, "best_bert_linear.pt"),
                                              map_location=DEVICE))
bert_linear_model.eval()
bl_preds, bl_labels_list = [], []
with torch.no_grad():
    for batch in bl_test_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labs = batch["labels"].to(DEVICE)
        res = bert_linear_model(ids, mask)
        bl_preds.extend(res["logits"].argmax(-1).cpu().numpy())
        bl_labels_list.extend(labs.cpu().numpy())

bl_yt, bl_yp = np.array(bl_labels_list), np.array(bl_preds)
bert_linear_metrics = {
    "accuracy": accuracy_score(bl_yt, bl_yp),
    "f1_macro": f1_score(bl_yt, bl_yp, average="macro"),
    "precision": precision_score(bl_yt, bl_yp, average="macro"),
    "recall": recall_score(bl_yt, bl_yp, average="macro"),
    "kappa": cohen_kappa_score(bl_yt, bl_yp),
}
print(f"\nBERT+Linear Results: {bert_linear_metrics}")
print("\nClassification Report (BERT+Linear):")
print(classification_report(bl_yt, bl_yp, target_names=LABEL_NAMES))
plot_confusion(bl_yt, bl_yp, title="BERT+Linear Baseline",
               save=os.path.join(config.output_dir, "cm_bert_linear.png"))
plot_curves(bl_history, save=os.path.join(config.output_dir, "curves_bert_linear.png"))

# Free GPU memory
del bert_linear_model
clear_gpu_cache()


In [ ]:
# ============================================================================
# CELL 16e: Aspect-Level Evaluation (ATE + ASC) — Full Report
# ============================================================================
# Final evaluation of ATE and ASC on SemEval-2014 test set.

print("=" * 60)
print("ASPECT-LEVEL EVALUATION (SemEval-2014 Test)")
print("=" * 60)

model.eval()
ate_preds_final, ate_labels_final = [], []
asc_preds_final, asc_labels_final = [], []
aspect_details = []  # for error analysis

with torch.no_grad():
    for batch in asp_test_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        bio = batch["bio_labels"].to(DEVICE)
        asc_lab = batch["asp_sentiment"].to(DEVICE)

        token_states = model.bert(input_ids=ids, attention_mask=mask).last_hidden_state
        ate_res = model.forward_ate(token_states, mask)

        for b in range(ids.shape[0]):
            real_len = mask[b].sum().item()
            pred_tags = ate_res["tags"][b][:real_len]
            true_tags = bio[b][:real_len].cpu().tolist()
            ate_preds_final.extend(pred_tags)
            ate_labels_final.extend(true_tags)

        # ASC — use CRF-predicted tags (not gold) for realistic evaluation
        cls_repr = token_states[:, 0, :]
        pred_bio_e = torch.full_like(bio, 2)  # default O
        for b in range(ids.shape[0]):
            rl = mask[b].sum().item()
            pt = ate_res["tags"][b][:rl]
            pred_bio_e[b, :rl] = torch.tensor(pt, device=DEVICE)
        bio_mask_eval = (pred_bio_e == 0) | (pred_bio_e == 1)
        asp_repr = torch.zeros_like(cls_repr)
        for b in range(ids.shape[0]):
            asp_tokens = token_states[b][bio_mask_eval[b]]
            if len(asp_tokens) > 0:
                asp_repr[b] = asp_tokens.mean(dim=0)
            else:
                asp_repr[b] = cls_repr[b]
        asc_res = model.forward_asc(asp_repr, cls_repr)
        asc_preds_final.extend(asc_res["asc_logits"].argmax(-1).cpu().numpy())
        asc_labels_final.extend(asc_lab.cpu().numpy())

# ATE metrics
print("\n--- Aspect Term Extraction (BIO Tagging) ---")
ate_report = classification_report(ate_labels_final, ate_preds_final,
                                    target_names=BIO_TAG_NAMES, output_dict=True)
print(classification_report(ate_labels_final, ate_preds_final, target_names=BIO_TAG_NAMES))

ate_metrics = {
    "ate_f1_macro": f1_score(ate_labels_final, ate_preds_final, average="macro"),
    "ate_f1_weighted": f1_score(ate_labels_final, ate_preds_final, average="weighted"),
    "ate_accuracy": accuracy_score(ate_labels_final, ate_preds_final),
    "ate_precision": precision_score(ate_labels_final, ate_preds_final, average="macro"),
    "ate_recall": recall_score(ate_labels_final, ate_preds_final, average="macro"),
    "ate_b_asp_f1": ate_report["B-ASP"]["f1-score"],
    "ate_i_asp_f1": ate_report["I-ASP"]["f1-score"],
}

# ASC metrics
print("\n--- Aspect Sentiment Classification ---")
asc_yt, asc_yp = np.array(asc_labels_final), np.array(asc_preds_final)
print(classification_report(asc_yt, asc_yp, target_names=LABEL_NAMES))

asc_metrics = {
    "asc_accuracy": accuracy_score(asc_yt, asc_yp),
    "asc_f1_macro": f1_score(asc_yt, asc_yp, average="macro"),
    "asc_precision": precision_score(asc_yt, asc_yp, average="macro"),
    "asc_recall": recall_score(asc_yt, asc_yp, average="macro"),
}

# Confusion matrices
plot_confusion(asc_yt, asc_yp, title="Aspect Sentiment (SemEval-2014)",
               save=os.path.join(config.output_dir, "cm_asc.png"))

# Save aspect reports
pd.DataFrame(ate_report).T.to_csv(os.path.join(config.output_dir, "classification_report_ate.csv"))
pd.DataFrame(classification_report(asc_yt, asc_yp, target_names=LABEL_NAMES, output_dict=True)).T.to_csv(
    os.path.join(config.output_dir, "classification_report_asc.csv"))

print(f"\nATE Summary: F1-macro={ate_metrics['ate_f1_macro']:.4f}, B-ASP F1={ate_metrics['ate_b_asp_f1']:.4f}")
print(f"ASC Summary: F1-macro={asc_metrics['asc_f1_macro']:.4f}, Accuracy={asc_metrics['asc_accuracy']:.4f}")


In [ ]:
# ============================================================================
# CELL 16f: Sentence-Level Evaluation
# ============================================================================
# Evaluates sentence-level sentiment using the HMGS sentence head.
# Uses SemEval sentences (which have known aspect sentiments) as proxy
# ground truth, plus Amazon test reviews decomposed into sentences.

print("=" * 60)
print("SENTENCE-LEVEL EVALUATION")
print("=" * 60)

model.eval()

# Strategy 1: Use SemEval sentences with known aspect sentiment as sentence GT
# If a sentence has all positive aspects -> sentence is positive, etc.
# This gives us ~800-1200 sentences with reliable labels.
sent_preds, sent_labels = [], []
sent_texts = []

with torch.no_grad():
    for item in semeval_test:
        ids = item["input_ids"].unsqueeze(0).to(DEVICE)
        mask = item["attention_mask"].unsqueeze(0).to(DEVICE)
        label = int(item["asp_sentiment"])  # use aspect sentiment as sentence proxy

        cls = model.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0, :]
        sent_res = model.forward_sent(cls)
        pred = sent_res["sent_logits"].argmax(-1).item()
        sent_preds.append(pred)
        sent_labels.append(label)
        sent_texts.append(item.get("text", ""))

sent_yt = np.array(sent_labels)
sent_yp = np.array(sent_preds)

sent_metrics = {
    "sent_accuracy": accuracy_score(sent_yt, sent_yp),
    "sent_f1_macro": f1_score(sent_yt, sent_yp, average="macro"),
    "sent_precision": precision_score(sent_yt, sent_yp, average="macro"),
    "sent_recall": recall_score(sent_yt, sent_yp, average="macro"),
    "sent_kappa": cohen_kappa_score(sent_yt, sent_yp),
}

print(f"\nSentence-Level Results (SemEval proxy, n={len(sent_yt)}):")
print(f"  Accuracy:  {sent_metrics['sent_accuracy']:.4f}")
print(f"  F1-Macro:  {sent_metrics['sent_f1_macro']:.4f}")
print(f"  Precision: {sent_metrics['sent_precision']:.4f}")
print(f"  Recall:    {sent_metrics['sent_recall']:.4f}")
print(f"  Kappa:     {sent_metrics['sent_kappa']:.4f}")

print("\nClassification Report (Sentence-Level):")
print(classification_report(sent_yt, sent_yp, target_names=LABEL_NAMES))

plot_confusion(sent_yt, sent_yp, title="Sentence-Level Sentiment",
               save=os.path.join(config.output_dir, "cm_sentence.png"))

# Strategy 2: Sentence-level on Amazon test reviews (distant supervision check)
# Measure how well sentence predictions agree with document labels.
print("\n--- Sentence-Document Agreement (Amazon Test, first 500 reviews) ---")
agreement_count, total_sents = 0, 0
with torch.no_grad():
    for idx in range(min(500, len(test_ds))):
        sample = test_ds[idx]
        doc_label = sample["doc_label"].item()
        ids = sample["input_ids"].unsqueeze(0).to(DEVICE)
        mask = sample["attention_mask"].unsqueeze(0).to(DEVICE)
        smask = sample["sent_mask"]
        n_sents = smask.sum().item()

        for s in range(n_sents):
            s_ids = ids[0, s:s+1, :]
            s_mask = mask[0, s:s+1, :]
            cls = model.bert(input_ids=s_ids, attention_mask=s_mask).last_hidden_state[:, 0, :]
            pred = model.forward_sent(cls)["sent_logits"].argmax(-1).item()
            if pred == doc_label:
                agreement_count += 1
            total_sents += 1

agreement_rate = agreement_count / max(total_sents, 1)
print(f"  Sentences evaluated: {total_sents}")
print(f"  Agreement with document label: {agreement_rate:.1%}")
print(f"  (Expected ~65-75% due to mixed-sentiment reviews)")

# Save
pd.DataFrame(classification_report(sent_yt, sent_yp, target_names=LABEL_NAMES, output_dict=True)).T.to_csv(
    os.path.join(config.output_dir, "classification_report_sentence.csv"))
print(f"\nSaved: classification_report_sentence.csv")

In [ ]:
# ============================================================================
# CELL 16g: Bootstrap Statistical Significance Testing
# ============================================================================
# Pairwise bootstrap test (Efron & Tibshirani, 1993) to determine whether
# differences between models are statistically significant (p < 0.05).
# Tests HMGS against each baseline on F1-macro.

from scipy import stats as sp_stats

def bootstrap_paired_test(y_true, preds_a, preds_b, metric_fn, n_boot=10000, seed=42):
    """
    Two-sided paired bootstrap test.
    Returns: (delta, p_value, ci_low, ci_high)
      delta   = metric(A) - metric(B) on full sample
      p_value = proportion of bootstrap samples where delta flips sign
      ci      = 95% confidence interval of delta
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    
    score_a = metric_fn(y_true, preds_a)
    score_b = metric_fn(y_true, preds_b)
    observed_delta = score_a - score_b
    
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.randint(0, n, size=n)
        yt_b = y_true[idx]
        pa_b = preds_a[idx]
        pb_b = preds_b[idx]
        deltas[i] = metric_fn(yt_b, pa_b) - metric_fn(yt_b, pb_b)
    
    # Two-sided p-value
    p_val = 2.0 * min(np.mean(deltas >= 0), np.mean(deltas <= 0))
    p_val = min(p_val, 1.0)
    
    ci_low = np.percentile(deltas, 2.5)
    ci_high = np.percentile(deltas, 97.5)
    
    return observed_delta, p_val, ci_low, ci_high

# Metric function for bootstrap
def f1_macro_metric(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

print("=" * 70)
print("BOOTSTRAP SIGNIFICANCE TESTING (10,000 resamples)")
print("=" * 70)

# Document-level: Compare HMGS against all baselines
comparisons = [
    ("HMGS vs TF-IDF+LR",            hmgs_yp, tfidf_yp),
    ("HMGS vs BERT+BiLSTM+Attn",     hmgs_yp, ab_yp),
    ("HMGS vs BERT+Linear",          hmgs_yp, bl_yp),
    ("BERT+BiLSTM+Attn vs TF-IDF+LR", ab_yp, tfidf_yp),
    ("BERT+Linear vs TF-IDF+LR",     bl_yp, tfidf_yp),
]

# Common ground truth for document-level
doc_gt = hmgs_yt  # all share the same test set

sig_results = {}
print(f"\n{'Comparison':<35} {'ΔF1':>8} {'p-value':>10} {'95% CI':>20} {'Sig?':>6}")
print("-" * 82)
for name, preds_a, preds_b in comparisons:
    # Ensure same length
    min_len = min(len(doc_gt), len(preds_a), len(preds_b))
    delta, p_val, ci_lo, ci_hi = bootstrap_paired_test(
        doc_gt[:min_len], preds_a[:min_len], preds_b[:min_len], f1_macro_metric
    )
    sig = "Yes ***" if p_val < 0.001 else "Yes **" if p_val < 0.01 else "Yes *" if p_val < 0.05 else "No"
    sig_results[name] = {"delta_f1": delta, "p_value": p_val, "ci_low": ci_lo, "ci_high": ci_hi, "significant": sig}
    print(f"{name:<35} {delta:>+8.4f} {p_val:>10.4f} [{ci_lo:>+.4f}, {ci_hi:>+.4f}] {sig:>6}")

print(f"\n{'='*82}")
print("Significance levels: * p<0.05, ** p<0.01, *** p<0.001")
print("Positive ΔF1 means the first model outperforms the second.")

# Save significance results
sig_df = pd.DataFrame(sig_results).T
sig_df.index.name = "Comparison"
sig_path = os.path.join(config.output_dir, "significance_tests.csv")
sig_df.to_csv(sig_path)
print(f"\nSaved: {sig_path}")

In [ ]:
# ============================================================================
# CELL 17: Attention Visualisation on sample reviews
# ============================================================================

samples = [
    "The phone has an amazing camera and the screen is beautiful. "
    "However, the battery life is terrible and it barely lasts half a day. "
    "Customer service was helpful when I reached out.",

    "Best purchase I've made this year. "
    "The build quality is outstanding and it works flawlessly.",

    "Terrible product. Broke after two days. "
    "Packaging was nice though.",
]

for i, rev in enumerate(samples):
    print(f"\n--- Review {i+1} ---")
    plot_doc_attention(model, rev, tokenizer, config,
                       save=os.path.join(config.output_dir, f"attn_{i+1}.png"))

In [ ]:
# ============================================================================
# CELL 18: Full multi-seed experiment (uncomment to run)
# ============================================================================
# This takes ~20-30 hours on T4. Reduce config.seeds for faster runs.

# results, histories = run_experiment(config, train_ds, val_ds, test_ds)

# # Save results
# import json
# with open(os.path.join(config.output_dir, "results.json"), "w") as f:
#     json.dump({k: v.tolist() for k, v in results.items()}, f, indent=2)

print("Full experiment cell ready. Uncomment to run.")

In [ ]:
# ============================================================================
# CELL 19: Results Comparison Table (All Models, All Granularity Levels)
# ============================================================================

# --- A. Document-Level Comparison (4 models) ---
print("\n" + "=" * 90)
print("TABLE 1: DOCUMENT-LEVEL SENTIMENT CLASSIFICATION")
print("=" * 90)

all_doc_metrics = {
    "TF-IDF + LR": tfidf_metrics,
    "BERT+Linear": bert_linear_metrics,
    "BERT+BiLSTM+Attn": ablation_metrics,
    "HMGS (Ours)": hmgs_metrics,
}

doc_keys = ["accuracy", "f1_macro", "precision", "recall", "kappa"]
header = f"{'Model':<25}" + "".join(f"{m:>12}" for m in doc_keys)
print(header)
print("-" * 85)
for name, m in all_doc_metrics.items():
    row = f"{name:<25}" + "".join(f"{m[k]:>12.4f}" for k in doc_keys)
    print(row)
print("=" * 85)

# Improvements
for bname in ["TF-IDF + LR", "BERT+Linear", "BERT+BiLSTM+Attn"]:
    delta = (hmgs_metrics['f1_macro'] - all_doc_metrics[bname]['f1_macro']) * 100
    print(f"HMGS vs {bname:<20s}: F1 improvement = {delta:+.2f}pp")

# --- B. Aspect-Level Results ---
print("\n" + "=" * 90)
print("TABLE 2: ASPECT-LEVEL EVALUATION (SemEval-2014)")
print("=" * 90)
print(f"  ATE F1 (Macro):    {ate_metrics['ate_f1_macro']:.4f}")
print(f"  ATE Precision:     {ate_metrics['ate_precision']:.4f}")
print(f"  ATE Recall:        {ate_metrics['ate_recall']:.4f}")
print(f"  ASC F1 (Macro):    {asc_metrics['asc_f1_macro']:.4f}")
print(f"  ASC Accuracy:      {asc_metrics['asc_accuracy']:.4f}")

# --- C. Sentence-Level Results ---
print("\n" + "=" * 90)
print("TABLE 3: SENTENCE-LEVEL EVALUATION")
print("=" * 90)
for k, v in sent_metrics.items():
    print(f"  {k:<20s}: {v:.4f}")

# --- D. Combined multi-granularity overview ---
print("\n" + "=" * 90)
print("TABLE 4: MULTI-GRANULARITY OVERVIEW (HMGS Model)")
print("=" * 90)
print(f"{'Granularity':<20} {'Metric':<20} {'Score':>10}")
print("-" * 52)
print(f"{'Document':<20} {'F1-Macro':<20} {hmgs_metrics['f1_macro']:>10.4f}")
print(f"{'Sentence':<20} {'F1-Macro':<20} {sent_metrics['sent_f1_macro']:>10.4f}")
print(f"{'Aspect (ATE)':<20} {'F1-Macro':<20} {ate_metrics['ate_f1_macro']:>10.4f}")
print(f"{'Aspect (ASC)':<20} {'F1-Macro':<20} {asc_metrics['asc_f1_macro']:>10.4f}")
print("=" * 52)


In [ ]:
# ============================================================================
# CELL 20: Export All Results for Thesis Documentation
# ============================================================================
# Saves: metrics CSV/JSON, training histories, predictions, aspect/sentence metrics

import csv
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Document-level metrics CSV
metrics_df = pd.DataFrame(all_doc_metrics).T
metrics_df.index.name = "Model"
metrics_path = os.path.join(config.output_dir, "results_comparison.csv")
metrics_df.to_csv(metrics_path)
print(f"Saved: {metrics_path}")
print(metrics_df.round(4).to_string())

# 2. Full results JSON (all granularities)
json_path = os.path.join(config.output_dir, "results_comparison.json")
with open(json_path, "w") as f:
    json.dump({
        "timestamp": timestamp,
        "dataset": "Amazon Customer Reviews 2023 (Electronics)",
        "train_size": len(df_train),
        "val_size": len(df_val),
        "test_size": len(df_test),
        "document_level": all_doc_metrics,
        "aspect_level": {"ate": ate_metrics, "asc": asc_metrics},
        "sentence_level": sent_metrics,
        "significance_tests": sig_results,
        "config": config.__dict__,
    }, f, indent=2, default=str)
print(f"Saved: {json_path}")

# 3. Training history CSVs
for hist_name, hist_data, hist_file in [
    ("HMGS", history, "hmgs_training_history.csv"),
    ("Ablation", ab_history, "ablation_training_history.csv"),
    ("BERT+Linear", bl_history, "bert_linear_training_history.csv"),
]:
    h_df = pd.DataFrame(hist_data)
    h_path = os.path.join(config.output_dir, hist_file)
    h_df.to_csv(h_path, index=False)
    print(f"Saved: {h_path}")

# 4. Per-class classification reports
for name, yt_arr, yp_arr in [
    ("tfidf", tfidf_yt, tfidf_yp),
    ("hmgs", hmgs_yt, hmgs_yp),
    ("ablation", ab_yt, ab_yp),
    ("bert_linear", bl_yt, bl_yp),
]:
    report = classification_report(yt_arr, yp_arr, target_names=LABEL_NAMES, output_dict=True)
    report_df = pd.DataFrame(report).T
    report_path = os.path.join(config.output_dir, f"classification_report_{name}.csv")
    report_df.to_csv(report_path)
    print(f"Saved: {report_path}")

# 5. Predictions CSV (for error analysis)
min_len = min(len(hmgs_yt), len(tfidf_yp), len(ab_yp), len(bl_yp))
pred_df = pd.DataFrame({
    "text": df_test["review_text"].values[:min_len],
    "true_label": hmgs_yt[:min_len],
    "hmgs_pred": hmgs_yp[:min_len],
    "tfidf_pred": tfidf_yp[:min_len],
    "ablation_pred": ab_yp[:min_len],
    "bert_linear_pred": bl_yp[:min_len],
    "true_name": [LABEL_NAMES[i] for i in hmgs_yt[:min_len]],
    "hmgs_name": [LABEL_NAMES[i] for i in hmgs_yp[:min_len]],
    "hmgs_correct": (hmgs_yt[:min_len] == hmgs_yp[:min_len]),
})
pred_path = os.path.join(config.output_dir, "predictions.csv")
pred_df.to_csv(pred_path, index=False)
print(f"Saved: {pred_path}")

print(f"\nAll results exported to: {config.output_dir}/")


In [ ]:
# ============================================================================
# CELL 21: List All Generated Output Files
# ============================================================================

print("Generated Outputs for Thesis Documentation")
print("=" * 65)

output_files = {
    "Figures": [
        "eda.png                    - EDA: star/sentiment/length distributions",
        "cm_tfidf.png               - Confusion matrix: TF-IDF baseline",
        "cm_hmgs.png                - Confusion matrix: HMGS model",
        "cm_ablation.png            - Confusion matrix: BERT+BiLSTM+Attn",
        "cm_bert_linear.png         - Confusion matrix: BERT+Linear baseline",
        "cm_sentence.png            - Confusion matrix: Sentence-level eval",
        "cm_asc.png                 - Confusion matrix: Aspect Sentiment (SemEval)",
        "curves_hmgs.png            - Training curves: HMGS (loss + F1)",
        "curves_ablation.png        - Training curves: BERT+BiLSTM+Attn",
        "curves_bert_linear.png     - Training curves: BERT+Linear",
        "attn_1.png                 - Attention visualisation: mixed review",
        "attn_2.png                 - Attention visualisation: positive review",
        "attn_3.png                 - Attention visualisation: negative review",
    ],
    "Tables / Data": [
        "results_comparison.csv     - 4-model doc-level comparison (thesis table)",
        "results_comparison.json    - All results + aspect/sentence (machine-readable)",
        "significance_tests.csv     - Bootstrap pairwise significance tests",
        "config.json                - All hyperparameters used",
    ],
    "Detailed Reports": [
        "classification_report_tfidf.csv        - Per-class P/R/F1: TF-IDF",
        "classification_report_hmgs.csv         - Per-class P/R/F1: HMGS",
        "classification_report_ablation.csv     - Per-class P/R/F1: BERT+BiLSTM",
        "classification_report_bert_linear.csv  - Per-class P/R/F1: BERT+Linear",
        "classification_report_ate.csv          - Per-class P/R/F1: Aspect Term Extraction",
        "classification_report_asc.csv          - Per-class P/R/F1: Aspect Sentiment",
        "classification_report_sentence.csv     - Per-class P/R/F1: Sentence-level",
    ],
    "Training Logs": [
        "hmgs_training_history.csv          - Epoch-by-epoch HMGS metrics",
        "ablation_training_history.csv      - Epoch-by-epoch ablation metrics",
        "bert_linear_training_history.csv   - Epoch-by-epoch BERT+Linear metrics",
    ],
    "Error Analysis": [
        "predictions.csv            - All test predictions (4 models) for error analysis",
    ],
    "Data Splits": [
        "train.csv                  - Training split",
        "val.csv                    - Validation split",
        "test.csv                   - Test split",
    ],
}

for section, files in output_files.items():
    print(f"\n{section}:")
    for f in files:
        print(f"  {config.output_dir}/{f}")

# Verify files exist
import glob as glob_mod
existing = glob_mod.glob(os.path.join(config.output_dir, "*"))
print(f"\nTotal files in {config.output_dir}/: {len(existing)}")
print("\nNotebook execution complete. All results ready for thesis documentation.")



In [ ]:
# ============================================================================
# CELL 22: Download All Outputs as ZIP
# ============================================================================
# Bundles every figure, CSV, JSON, and report into a single ZIP file.
# On Colab the file auto-downloads; locally it just creates the archive.

import zipfile, glob as _glob, os as _os

ZIP_NAME = "thesis_outputs.zip"
out_dir  = config.output_dir  # ./outputs

files = sorted(_glob.glob(_os.path.join(out_dir, "*")))
if not files:
    print("No output files found. Run all cells above first.")
else:
    with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
        for fp in files:
            arcname = _os.path.join("thesis_outputs", _os.path.basename(fp))
            zf.write(fp, arcname)

    size_mb = _os.path.getsize(ZIP_NAME) / 1e6
    print(f"Created: {ZIP_NAME}  ({size_mb:.1f} MB, {len(files)} files)")
    print(f"\nContents:")
    for fp in files:
        ext = _os.path.splitext(fp)[1]
        tag = {'.png':'[FIG]', '.csv':'[CSV]', '.json':'[JSON]'}.get(ext, '[FILE]')
        print(f"  {tag}  {_os.path.basename(fp)}")

    # Auto-download on Colab
    try:
        from google.colab import files as colab_files
        colab_files.download(ZIP_NAME)
        print(f"\nDownload started automatically.")
    except ImportError:
        print(f"\nNot running on Colab. ZIP saved to: {_os.path.abspath(ZIP_NAME)}")

---
## Summary

### Models Evaluated (Document-Level)
| # | Model | Type | Purpose |
|---|-------|------|---------|
| 1 | TF-IDF + Logistic Regression | Traditional ML | Lower bound baseline |
| 2 | BERT + Linear | Deep Learning | Minimal BERT baseline |
| 3 | BERT + BiLSTM + Attention | Deep Learning | Original thesis architecture (ablation) |
| 4 | **HMGS (Ours)** | Multi-task Deep Learning | Proposed hierarchical multi-granularity model |

### Multi-Granularity Evaluation
| Level | Method | Data Source |
|-------|--------|-----------|
| Document | 4-model comparison (F1, Acc, P, R, Kappa) | Amazon Reviews 2023 |
| Sentence | Proxy evaluation via SemEval + agreement analysis | SemEval-2014 + Amazon |
| Aspect (ATE) | BIO sequence labelling F1 | SemEval-2014 Task 4 |
| Aspect (ASC) | 3-class sentiment F1 on extracted aspects | SemEval-2014 Task 4 |

### Statistical Rigour
- **Bootstrap significance testing** (10,000 resamples) for all pairwise comparisons
- **5-seed experiment** available (Cell 18) for mean \u00b1 std reporting
- **Per-class classification reports** for all models and granularity levels

### Key Outputs
- **12 figures** saved as high-resolution PNG (EDA, confusion matrices, training curves, attention maps)
- **7 classification reports** (4 doc-level + ATE + ASC + sentence)
- **Results comparison table** in CSV and JSON with all granularity levels
- **Significance tests** with p-values and 95% confidence intervals
- **Predictions file** for error analysis (all 4 models)
- **Training histories** for learning curve analysis (3 deep learning models)
- **Data splits** for reproducibility

### Execution Notes
- **Quick run** (Cells 15–32): ~3–5 hours on T4 GPU with 50K training samples
- **Full multi-seed** (Cell 18): Uncomment to run 5-seed experiment for mean \u00b1 std reporting
- **CPU fallback**: Works on CPU but significantly slower; reduce `train_size` to 5000 for testing
- All outputs saved to `./outputs/` directory
